# Step 3 — Retriever/Generator wrapper verification

Step 1 (`00_smoke_test.ipynb`) proved the underlying config works by hand-building
everything inline: raw `AutoModelForCausalLM.from_pretrained`, manually attached
`PeftModel` adapters, directly-constructed `VDocRetriever`/`VDocGenerator` objects.

This notebook tests the *actual production code* — `vdocrag_app.model_manager.ModelManager`,
`vdocrag_app.retriever.VDocRetrieverWrapper`, `vdocrag_app.generator.VDocGeneratorWrapper`
— the classes `app.py` actually calls. All three confirmed Step 1 fixes (eager attention via
config object, the three `transformers` compat shims, `num_crops=4`, `use_cache=False` for
generation) are baked into `ModelManager.setup()` and `VDocGeneratorWrapper.answer()` now,
so none of that should need to be repeated here — if it does, that's a real bug in how the
wrapper classes were built, which is exactly what this notebook exists to catch.

**Pass criteria**: results should be *close to* Step 1's confirmed-working raw output
(similarity ordering correct, generation answer in the same ballpark as `"11.4 million"`)
— not identical to the decimal (different code paths, e.g. `encode_documents_batch`'s
per-image loop vs. Step 1's inline loop, can produce tiny floating-point differences),
but the *behavior* should match. A meaningfully different result here means the wrapper
classes diverge from the confirmed-working configuration somewhere — a real bug to find,
not a quantization artifact to shrug off.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.environ['HF_HOME'] = '/content/drive/MyDrive/vdocrag-project/hf_cache'
os.makedirs(os.environ['HF_HOME'], exist_ok=True)

In [ ]:
!pip install -q --upgrade pip
# pinned to the last 4.x release before transformers' 5.0 major version --
# see docs/implementation_plan.md Section 4.6b/4.6d/4.6e/4.6f for why.
!pip install -q transformers==4.57.3 accelerate bitsandbytes peft pillow

# NTT's package installed by URL every session, never vendored -- see docs/licenses.md
!pip install -q git+https://github.com/nttmdlab-nlp/VDocRAG.git

!rm -rf /content/repo
!git clone https://github.com/thejainamjain/vdocrag-project.git /content/repo
import sys
sys.path.insert(0, '/content/repo')

In [ ]:
from vdocrag_app.telemetry import setup_logging
logger = setup_logging('step3_wrapper_test')

## Load everything through ModelManager (not hand-built)

This single call replaces Step 1's entire Check 1 + Check 2 + the `num_crops`/shim
fixes — if this cell fails, the bug is in how those fixes got translated into
`model_manager.py`, not a new problem.

In [ ]:
import torch
from vdocrag_app.model_manager import ModelManager, ModelManagerConfig

manager = ModelManager(ModelManagerConfig())  # defaults: eager, shared adapters, num_crops=4
manager.setup()

print(f"Mode: {manager.mode}")
print(f"VRAM: {manager.vram_report()}")
print("PASS if no exception and VRAM is in the low single-digit GB range (Step 1 confirmed ~2.44-2.5GB).")

## Wrap with the actual retriever/generator classes

In [ ]:
from vdocrag_app.retriever import VDocRetrieverWrapper
from vdocrag_app.generator import VDocGeneratorWrapper

retriever = VDocRetrieverWrapper(manager)
generator = VDocGeneratorWrapper(manager)
print("Wrappers constructed.")

## Retrieval: encode_query / encode_documents_batch

Same two example queries/images as Step 1's Check 3a, so the results are directly
comparable to a known-good baseline.

In [ ]:
import requests
from io import BytesIO
from PIL import Image
import numpy as np

queries = [
    "What is the total percentage of Palestinians residing at West Bank?",
    "How many international visitors came to Japan in 2017?",
]
urls = [
    "https://huggingface.co/datasets/NTT-hil-insight/OpenDocVQA/resolve/main/image1.png",
    "https://huggingface.co/datasets/NTT-hil-insight/OpenDocVQA/resolve/main/image2.png",
]
raw_images = [Image.open(BytesIO(requests.get(u).content)) for u in urls]

query_embeddings = np.stack([retriever.encode_query(q) for q in queries])
doc_embeddings = retriever.encode_documents_batch(raw_images)

def cosine(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

for i, q in enumerate(queries):
    sims = [cosine(query_embeddings[i], doc_embeddings[j]) for j in range(len(doc_embeddings))]
    print(f"Query {i} ({q[:40]}...): {sims}")

print("\nStep 1's confirmed-working raw-code baseline (num_crops=4): [0.523, 0.406] and [0.398, 0.586]")
print("PASS if ordering matches (query 0 -> image1 higher, query 1 -> image2 higher) and values are close to baseline --")
print("not identical (different code path: batch loop here vs. Step 1's inline loop), but close.")

## Generation: VDocGeneratorWrapper.answer()

This exercises the confirmed `use_cache=False` fix automatically -- no manual
workaround needed here, unlike Step 1 where it was discovered live.

In [ ]:
answer = generator.answer("How many international visitors came to Japan in 2017?", raw_images)
print("Answer:", answer)
print("\nStep 1's confirmed-working raw-code baseline (num_crops=4): '11.4 million'")
print("NTT's own reported output (flash_attention_2, A100, num_crops=16): '28.69m'")
print("PASS if this is close to the Step 1 baseline -- a meaningfully different answer means")
print("the wrapper's prompt construction or generation args diverge from what was confirmed working.")

## Results summary

Same pattern as Step 1 -- pulls the logged timing/VRAM data for these wrapper-level
calls specifically, which is more representative of real app.py usage than Step 1's
hand-rolled loop timings were.

In [ ]:
import pandas as pd
df = pd.read_json('/content/drive/MyDrive/vdocrag-project/logs/step3_wrapper_test.jsonl', lines=True)
cols = ['component', 'function', 'duration_ms', 'vram_before_gb', 'vram_after_gb', 'vram_peak_gb', 'status']
display(df[df.component.isin(['retriever', 'generator', 'model_manager'])][cols])